# FractalMSF: Multi-Scale Fragment Training
## Learning to Generate Full High-Res Images from Scattered Multi-Scale Patches

**Colab Training Notebook**  
Project: `~/Desktop/fractal-msf/`  
Based on: [Fractal Generative Models](https://github.com/LTH14/fractalgen) (Li et al., 2025)

---

### Experiment Pipeline

```
Step 1: Setup (clone repo, install deps)
Step 2: Prepare MSFT data (scattered multi-scale patches)
Step 3: Phase 1 — Per-level pretraining
Step 4: Phase 2 — End-to-end alignment
Step 5: DFS Generation — Generate full high-res images
Step 6: Evaluation — FID, visual inspection
```

### MSFT Data Setup

| Level | Data | Spatial Coverage | Resolution |
|---|---|---|---|
| D₂ (top) | Full image ↓16 | 256×256 area | 16×16 effective |
| D₁ (mid) | 64×64 region ↓4 | ~6% area | 16×16 effective |
| D₀ (bottom) | 16×16 patch, raw | ~1% area | 16×16 (full sharpness) |

---
## Step 1: Environment Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

In [ ]:
# Clone FractalGen (base model)
!git clone https://github.com/LTH14/fractalgen.git /content/fractalgen

# Clone FractalMSF (our code)
!git clone https://github.com/SpencerRaw/fractal-msf.git /content/fractal-msf 2>/dev/null || echo "(local copy, skip clone)"

# If no git repo yet, create from local files:
!mkdir -p /content/fractal-msf/code/msft

In [ ]:
# Install dependencies (matches FractalGen's environment)
!pip install -q timm torch_fidelity scipy tensorboard

# Check timm version
import timm
print(f"timm version: {timm.__version__}")

In [ ]:
import sys
import os

# Add paths
FRACTALGEN_PATH = '/content/fractalgen'
MSFT_PATH = '/content/fractal-msf/code/msft'

sys.path.insert(0, FRACTALGEN_PATH)
sys.path.insert(0, MSFT_PATH)

# Output directory for checkpoints
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"FractalGen path: {FRACTALGEN_PATH}")
print(f"MSFT path: {MSFT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

---
## Step 2: MSFT Dataset

Copy our MSFT dataset code to Colab, then download ImageNet 64×64.

In [ ]:
%%writefile /content/fractal-msf/code/msft/msft_dataset.py
"""MSFT Dataset: Multi-Scale Fragment Training."""

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import numpy as np


class MSFTDataset(Dataset):
    def __init__(
        self, root, img_size=64, patch_size=16,
        n_highres=5, n_midres=2,
        midres_span=32, midres_downsample=2,
        full_downsample=4, train=True,
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_highres = n_highres
        self.n_midres = n_midres
        self.midres_span = midres_span
        self.midres_downsample = midres_downsample
        self.full_downsample = full_downsample
        self.train = train
        
        self.norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.norm_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        
        split = 'train' if train else 'val'
        dataset_path = f"{root}/{split}"
        
        transform = transforms.Compose([
            transforms.RandomHorizontalFlip() if train else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
        ])
        self.dataset = ImageFolder(dataset_path, transform=transform)

    def __len__(self):
        return len(self.dataset)

    def _norm(self, img):
        return (img - self.norm_mean) / self.norm_std

    def _crop(self, img, size):
        _, h, w = img.shape
        if h == size and w == size:
            return img
        top = np.random.randint(0, h - size + 1)
        left = np.random.randint(0, w - size + 1)
        return img[:, top:top+size, left:left+size]

    def _down(self, img, factor):
        return F.interpolate(img.unsqueeze(0), scale_factor=1.0/factor, mode='area').squeeze(0)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = self._norm(img)
        result = {'label': label}
        
        # D₂: 低清全图
        result['full_low'] = self._down(img, self.full_downsample)
        
        # D₁: 中分辨局部
        midres_list = []
        for _ in range(self.n_midres):
            r = self._crop(img, self.midres_span)
            midres_list.append(self._down(r, self.midres_downsample))
        result['midres'] = torch.stack(midres_list) if midres_list else torch.empty(0)
        
        # D₀: 高清局部
        highres_list = []
        for _ in range(self.n_highres):
            highres_list.append(self._crop(img, self.patch_size))
        result['highres'] = torch.stack(highres_list) if highres_list else torch.empty(0)
        
        return result


def msft_collate_fn(batch):
    """Collate list of dicts into batched dict."""
    result = {}
    for key in batch[0].keys():
        values = [item[key] for item in batch]
        if isinstance(values[0], torch.Tensor):
            if values[0].dim() == 3:
                result[key] = torch.stack(values)
            elif values[0].dim() == 4:
                result[key] = torch.cat(values, dim=0)
        else:
            result[key] = torch.tensor(values)
    return result

In [ ]:
# ═══════════════════════════════════════════════
# Download ImageNet 64×64
# ═══════════════════════════════════════════════

import os, shutil

COLAB_DATA = '/content/data/imagenet64'
os.makedirs(COLAB_DATA, exist_ok=True)

# ═══════════════════════════════════════════
# PATH A: Google Drive (if you pre-uploaded)
# ═══════════════════════════════════════════
USE_DRIVE = True  # ← Set False to download from HuggingFace

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_PATH = '/content/drive/MyDrive/imagenet64'
    if os.path.exists(f'{DRIVE_PATH}/train'):
        print(f'Copying from Drive...')
        # Copy to local disk (much faster I/O than reading from Drive directly)
        if not os.path.exists(f'{COLAB_DATA}/train'):
            !cp -r "$DRIVE_PATH"/* "$COLAB_DATA"/
        print('✓ Copied from Drive')
    else:
        print(f'⚠️  {DRIVE_PATH}/train not found!')
        print('   Upload ImageNet64 to your Google Drive first.')
        print('   Or set USE_DRIVE=False to download from HuggingFace.')

else:
    # ═══════════════════════════════════════════
    # PATH B: HuggingFace → download & resize to 64×64
    # ═══════════════════════════════════════════
    print('Downloading ImageNet from HuggingFace (resizing to 64×64)...')
    print('This downloads ~50 GB of 256×256 images and resizes to 64×64.')
    print('Estimated time: 30-60 min on Colab. Grab a coffee ☕')
    
    !pip install -q datasets pillow
    
    from datasets import load_dataset
    from PIL import Image
    import numpy as np
    
    # Download ImageNet-1k from HuggingFace
    # Use ILSVRC/imagenet-1k (official, ~150GB) or benjamin-paine (256×256, ~50GB)
    dataset = load_dataset('benjamin-paine/imagenet-1k-256x256', split='train', streaming=True)
    
    # Create ImageFolder structure
    class_names = set()
    count = 0
    max_per_class = 100  # Limit for quick testing; set to None for full dataset
    
    for sample in dataset:
        img = sample['image']  # PIL Image, 256×256
        label = sample['label']
        
        # Resize to 64×64
        img_64 = img.resize((64, 64), Image.LANCZOS)
        
        # Save to train/<label>/
        class_dir = f'{COLAB_DATA}/train/{label}'
        os.makedirs(class_dir, exist_ok=True)
        img_64.save(f'{class_dir}/{count:08d}.JPEG')
        
        count += 1
        if count % 1000 == 0:
            print(f'  Downloaded {count} images...')
        if max_per_class and count >= max_per_class * 1000:
            break
    
    print(f'✓ Downloaded {count} images to {COLAB_DATA}/train/')

# ═══════════════════════════════════════════
# Verify
# ═══════════════════════════════════════════
DATA_DIR = COLAB_DATA
print(f'\nData directory: {DATA_DIR}')
if os.path.exists(f'{DATA_DIR}/train'):
    n_classes = len(os.listdir(f'{DATA_DIR}/train'))
    n_images = sum(len(os.listdir(f'{DATA_DIR}/train/{c}')) for c in os.listdir(f'{DATA_DIR}/train'))
    print(f'✓ ImageNet64: {n_classes} classes, {n_images} images')
else:
    print('⚠️  Data not ready. Check the cell above for errors.')


In [ ]:
# Test MSFT dataset (64×64, 4×4 base patches)
from msft_dataset import MSFTDataset, msft_collate_fn
from torch.utils.data import DataLoader

dataset = MSFTDataset(
    root=DATA_DIR,
    img_size=64,
    patch_size=4,
    n_highres=8,
    n_midres=3,
    midres_span=16,
    midres_downsample=4,
    full_downsample=16,
    train=True,
)

loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=msft_collate_fn)

batch = next(iter(loader))
print("Batch keys:", list(batch.keys()))
print(f"  D₂ full_low:  {batch['full_low'].shape}")     # (B, 3, 4, 4) — 16 tokens
print(f"  D₁ midres:    {batch['midres'].shape}")       # (B*n, 3, 4, 4) — 16 tokens each
print(f"  D₀ highres:   {batch['highres'].shape}")      # (B*m, 3, 4, 4) — 16 tokens each
print(f"  label:        {batch['label']}")

# All three levels = 4×4×3 = 48 values → 16 spatial tokens × 3 channels
print("\n✓ All levels: 4×4 = 16 tokens. MSFT dataset working!")


---
## Step 3: Phase 1 — Per-Level Pretraining

Train each level of FractalGen independently with its corresponding scale data.

```
g₂ (top):    D₂ low-res full images  →  learn coarse structure
g₁ (mid):    D₁ mid-res regions      →  learn medium details  
g₀ (bottom): D₀ high-res patches     →  learn fine details
```

In [ ]:
import torch
from models import fractalgen

# ═══════════════════════════════════════════════
# Custom FractalMAR for 64×64 with 4×4 base patches
# All 3 levels have exactly 4×4 = 16 tokens
# img_size_list = (64, 16, 4, 1):
#   Level 0: 64÷16=4 → 4×4 grid = 16 tokens (D₂: full ↓16)
#   Level 1: 16÷4=4  → 4×4 grid = 16 tokens (D₁: 16×16 ↓4)
#   Level 2: 4÷1=4   → 4×4 grid = 16 tokens (D₀: 4×4 raw)
#   Level 3: pixel loss
# ═══════════════════════════════════════════════

model = fractalgen.FractalGen(
    img_size_list=(64, 16, 4, 1),
    embed_dim_list=(512, 256, 128, 64),       # smaller: only 16 tokens/level
    num_blocks_list=(12, 6, 3, 1),
    num_heads_list=(8, 4, 2, 4),
    generator_type_list=("mar", "mar", "mar", "ar"),
    label_drop_prob=0.1,
    class_num=1000,
    attn_dropout=0.1,
    proj_dropout=0.1,
    fractal_level=0,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FractalMAR-64-4x4: {n_params/1e6:.1f}M params")
print(f"  Level 0 (D₂): seq_len={(64//16)**2} tokens")
print(f"  Level 1 (D₁): seq_len={(16//4)**2} tokens")
print(f"  Level 2 (D₀): seq_len={(4//1)**2} tokens")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"\nDevice: {device}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Train top level (g₂): learn from D₂ (4×4 low-res full images)
# ═══════════════════════════════════════════════════════════

from msft_dataset import MSFTDataset, msft_collate_fn
from torch.utils.data import DataLoader

dataset_top = MSFTDataset(
    root=DATA_DIR, img_size=64, patch_size=4,
    n_highres=0, n_midres=0,
    full_downsample=16, train=True,
)
loader_top = DataLoader(dataset_top, batch_size=64, shuffle=True,
                        num_workers=2, pin_memory=True,
                        collate_fn=msft_collate_fn)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, betas=(0.9, 0.95), weight_decay=0.05)

print("Training top level (g₂) on 4×4 low-res full images...")
print(f"  D₂: 64×64 ↓16 = 4×4 patches, 16 tokens")

model.train()
EPOCHS_TOP = 100

for epoch in range(EPOCHS_TOP):
    total_loss = 0
    for step, batch in enumerate(loader_top):
        imgs = batch['full_low'].to(device)   # (B, 3, 4, 4)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = model(imgs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader_top)
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS_TOP} | Loss: {avg_loss:.4f}")

print(f"\n✓ Top level done! Final loss: {avg_loss:.4f}")
torch.save({'model': model.state_dict()}, f'{OUTPUT_DIR}/pretrain_top.pth')


In [ ]:
# ═══════════════════════════════════════════════════════════
# Train bottom level (g₀): learn from D₀ (4×4 high-res patches)
# ═══════════════════════════════════════════════════════════

dataset_bottom = MSFTDataset(
    root=DATA_DIR, img_size=64, patch_size=4,
    n_highres=8, n_midres=0,
    full_downsample=16, train=True,
)
loader_bottom = DataLoader(dataset_bottom, batch_size=32, shuffle=True,
                           num_workers=2, pin_memory=True,
                           collate_fn=msft_collate_fn)

print("Training bottom level (g₀) on 4×4 high-res patches...")
print(f"  D₀: 4×4 at full sharpness, 16 tokens")

model.train()
EPOCHS_BOTTOM = 100

for epoch in range(EPOCHS_BOTTOM):
    total_loss = 0
    for step, batch in enumerate(loader_bottom):
        imgs = batch['highres'].to(device)  # (B, N, 3, 4, 4)
        B, N = imgs.shape[:2]
        imgs = imgs.view(B * N, *imgs.shape[2:])
        labels = batch['label'].repeat_interleave(N).to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = model(imgs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader_bottom)
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS_BOTTOM} | Loss: {avg_loss:.4f}")

print(f"\n✓ Bottom level done! Final loss: {avg_loss:.4f}")
torch.save({'model': model.state_dict()}, f'{OUTPUT_DIR}/pretrain_bottom.pth')


---
## Step 4: Phase 2 — End-to-End Alignment

Fine-tune all levels together with paired data (D₀ ⊂ D₁ ⊂ D₂).

In [ ]:
%%writefile /content/fractal-msf/code/msft/paired_dataset.py
"""Paired MSFT dataset for Phase 2 end-to-end alignment.

Supports three pairing modes:
  'same_image': D0, D1, D2 from the same image (spatially aligned)
  'same_class': D0, D1, D2 from different images of same class
  'random':     D0, D1, D2 from random images

'same_class' mirrors MD: different simulation runs of the same system.

Default (64×64, 4×4 base):
  D0: 4×4 high-res patch, D1: 16×16↓4=4×4, D2: 64×64↓16=4×4
  All levels: 4×4 = 16 tokens
"""

import torch, torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import numpy as np

class MSFTPairedDataset(Dataset):
    """D0, D1, D2 fragments with configurable pairing mode."""
    
    def __init__(self, root, img_size=64, patch_size=4,
                 midres_span=16, midres_downsample=4, full_downsample=16,
                 train=True, pairing_mode='same_image'):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.midres_span = midres_span
        self.midres_downsample = midres_downsample
        self.full_downsample = full_downsample
        self.pairing_mode = pairing_mode
        
        self.norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.norm_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        
        split = 'train' if train else 'val'
        transform = transforms.Compose([transforms.ToTensor()])
        self.dataset = ImageFolder(f"{root}/{split}", transform=transform)
        
        if pairing_mode == 'same_class':
            self.class_to_indices = {}
            for j, (_, lbl) in enumerate(self.dataset.samples):
                self.class_to_indices.setdefault(lbl, []).append(j)

    def __len__(self): return len(self.dataset)

    def _norm(self, img):
        return (img - self.norm_mean.to(img.device)) / self.norm_std.to(img.device)

    def _down(self, img, factor):
        return F.interpolate(img.unsqueeze(0), scale_factor=1.0/factor, mode='area').squeeze(0)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = self._norm(img)
        result = {'label': label}
        total = len(self.dataset)

        if self.pairing_mode == 'same_image':
            result['full_low'] = self._down(img, self.full_downsample)
            grid = self.img_size // self.midres_span
            gi, gj = np.random.randint(0, grid, 2)
            y1, y2 = gi*self.midres_span, (gi+1)*self.midres_span
            x1, x2 = gj*self.midres_span, (gj+1)*self.midres_span
            region = img[:, y1:y2, x1:x2]
            result['midres'] = self._down(region, self.midres_downsample)
            sub = self.midres_span // self.patch_size
            si, sj = np.random.randint(0, sub, 2)
            py1, py2 = si*self.patch_size, (si+1)*self.patch_size
            px1, px2 = sj*self.patch_size, (sj+1)*self.patch_size
            result['highres'] = region[:, py1:py2, px1:px2]

        elif self.pairing_mode == 'same_class':
            sc = self.class_to_indices[label]
            idx2 = sc[np.random.randint(len(sc))]
            result['full_low'] = self._down(self._norm(self.dataset[idx2][0]), self.full_downsample)
            idx1 = sc[np.random.randint(len(sc))]
            img1 = self._norm(self.dataset[idx1][0])
            y1 = np.random.randint(0, img1.shape[1]-self.midres_span+1)
            x1 = np.random.randint(0, img1.shape[2]-self.midres_span+1)
            result['midres'] = self._down(img1[:,y1:y1+self.midres_span,x1:x1+self.midres_span], self.midres_downsample)
            idx0 = sc[np.random.randint(len(sc))]
            img0 = self._norm(self.dataset[idx0][0])
            py1 = np.random.randint(0, img0.shape[1]-self.patch_size+1)
            px1 = np.random.randint(0, img0.shape[2]-self.patch_size+1)
            result['highres'] = img0[:,py1:py1+self.patch_size,px1:px1+self.patch_size]

        else:  # random
            idx2 = np.random.randint(total)
            result['full_low'] = self._down(self._norm(self.dataset[idx2][0]), self.full_downsample)
            idx1 = np.random.randint(total)
            img1 = self._norm(self.dataset[idx1][0])
            y1 = np.random.randint(0, img1.shape[1]-self.midres_span+1)
            x1 = np.random.randint(0, img1.shape[2]-self.midres_span+1)
            result['midres'] = self._down(img1[:,y1:y1+self.midres_span,x1:x1+self.midres_span], self.midres_downsample)
            idx0 = np.random.randint(total)
            img0 = self._norm(self.dataset[idx0][0])
            py1 = np.random.randint(0, img0.shape[1]-self.patch_size+1)
            px1 = np.random.randint(0, img0.shape[2]-self.patch_size+1)
            result['highres'] = img0[:,py1:py1+self.patch_size,px1:px1+self.patch_size]

        return result


In [ ]:
# ═══════════════════════════════════════════════════════════
# Phase 2: End-to-End Alignment (64×64, 4×4 patches)
# ═══════════════════════════════════════════════════════════

from paired_dataset import MSFTPairedDataset

PAIRING_MODE = 'same_class'  # 'same_image' | 'same_class' | 'random'
print(f"Pairing mode: {PAIRING_MODE}")

dataset_paired = MSFTPairedDataset(
    root=DATA_DIR, img_size=64, patch_size=4,
    midres_span=16, midres_downsample=4, full_downsample=16, train=True,
    pairing_mode=PAIRING_MODE,
)
loader_paired = DataLoader(dataset_paired, batch_size=64, shuffle=True,
                           num_workers=2, pin_memory=True)

print(f"  D₂: 64×64↓16=4×4 | D₁: 16×16↓4=4×4 | D₀: 4×4 raw")
print(f"  All levels: 16 tokens")

model.train()
EPOCHS_ALIGN = 100

for epoch in range(EPOCHS_ALIGN):
    total_loss = 0
    for step, batch in enumerate(loader_paired):
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        
        losses = []
        
        with torch.cuda.amp.autocast():
            losses.append(model(batch['full_low'].to(device), labels))
        with torch.cuda.amp.autocast():
            losses.append(model(batch['midres'].to(device), labels))
        with torch.cuda.amp.autocast():
            losses.append(model(batch['highres'].to(device), labels))
        
        loss = sum(losses) / len(losses)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader_paired)
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS_ALIGN} | Loss: {avg_loss:.4f}")

print(f"\n✓ Alignment done! Final loss: {avg_loss:.4f}")
torch.save({'model': model.state_dict()}, f'{OUTPUT_DIR}/aligned.pth')


---
## Step 5: DFS Generation

Generate full high-resolution images from the trained model using depth-first traversal.

In [ ]:
import torch
from functools import partial

model.eval()

NUM_IMAGES = 16
class_labels = torch.randint(0, 1000, (NUM_IMAGES,)).to(device)

# FractalGen sample parameters
num_iter_list = [64, 16]  # 64 iterations for g₁ (16×16 patches), 16 for g₀ (4×4 pixels)
cfg = 1.0
temperature = 1.0

print(f"Generating {NUM_IMAGES} images...")
print(f"  num_iter_list: {num_iter_list}")
print(f"  cfg: {cfg}, temperature: {temperature}")

with torch.no_grad():
    with torch.cuda.amp.autocast():
        images = model.sample(
            cond_list=class_labels,
            num_iter_list=num_iter_list,
            cfg=cfg,
            cfg_schedule='linear',
            temperature=temperature,
            filter_threshold=1e-4,
            fractal_level=0,
            visualize=False,
        )

print(f"Generated images shape: {images.shape}")
print(f"  (should be {NUM_IMAGES}, 3, 64, 64)")

In [ ]:
# Visualize generated images
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Denormalize: (x * std) + mean
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

images_vis = images.cpu() * std + mean
images_vis = torch.clamp(images_vis, 0, 1)

# Make grid
grid = vutils.make_grid(images_vis, nrow=4, padding=2)

plt.figure(figsize=(12, 12))
plt.imshow(grid.permute(1, 2, 0))
plt.axis('off')
plt.title(f'FractalMSF Generated Images (64×64)')
plt.show()

# Save
vutils.save_image(grid, f'{OUTPUT_DIR}/generated_samples.png')
print(f"Saved to {OUTPUT_DIR}/generated_samples.png")

---
## Step 6: Evaluation — FID Score

Compute FID against the validation set.

In [ ]:
# Generate more images for FID evaluation
NUM_EVAL = 5000
print(f"Generating {NUM_EVAL} images for FID evaluation...")

all_images = []
batch_size = 64

model.eval()
with torch.no_grad():
    for start in range(0, NUM_EVAL, batch_size):
        end = min(start + batch_size, NUM_EVAL)
        bsz = end - start
        
        labels = torch.randint(0, 1000, (bsz,)).to(device)
        
        with torch.cuda.amp.autocast():
            imgs = model.sample(
                cond_list=labels,
                num_iter_list=[64, 16],
                cfg=1.0, cfg_schedule='linear',
                temperature=1.0, filter_threshold=1e-4,
                fractal_level=0, visualize=False,
            )
        all_images.append(imgs.cpu())
        
        if (start // batch_size) % 10 == 0:
            print(f"  {end}/{NUM_EVAL}")

gen_images = torch.cat(all_images, dim=0)
print(f"Generated {gen_images.shape[0]} images")

# Save for FID computation
gen_dir = f'{OUTPUT_DIR}/generated_for_fid'
os.makedirs(gen_dir, exist_ok=True)
for i in range(min(NUM_EVAL, gen_images.shape[0])):
    img = (gen_images[i] * std + mean).clamp(0, 1)
    img_pil = transforms.ToPILImage()(img)
    img_pil.save(f'{gen_dir}/{i:05d}.png')

print(f"Saved generated images to {gen_dir}")

# Compute FID
from torch_fidelity import calculate_metrics

metrics = calculate_metrics(
    input1=gen_dir,
    input2=f'{DATA_DIR}/val',
    cuda=True,
    isc=False,
    fid=True,
    verbose=True,
)

print(f"\n{'='*50}")
print(f"FID: {metrics['frechet_inception_distance']:.2f}")
print(f"{'='*50}")

---
## Results Summary

Record your results here as you run experiments.

| Experiment | Level Config | Epochs | FID↓ | Notes |
|---|---|---|---|---|
| Baseline (Oracle) | Full data, 3-level | 400 | ~3.15 | Original FractalMAR paper |
| MSFT 2-level (ours) | D₀ + D₂ | ? | ? | No mid-level |
| MSFT 3-level (ours) | D₀ + D₁ + D₂ | ? | ? | Full MSFT |
| SuperRes Only | D₂ only | ? | ? | Traditional super-resolution |

### Key Questions

1. Can MSFT generate recognizable images? (FID < 30)
2. Does 3-level beat 2-level? (FID improvement > 2 points)
3. How close to Oracle? (within 2× FID = success)

### Next Steps After Colab

- Transfer to cloud GPU (A100/H100) for full 400-epoch training
- Run 256×256 experiments
- Systematic ablation: coverage, training protocol, level count